# Compare Orbit NC with Real L3 SSH Pass

Validates orbit.nc geometry against real L3 SSH data from local fcollections database.

In [ ]:
import pathlib
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from fcollections.implementations import NetcdfFilesDatabaseSwotLRL3

%matplotlib inline
plt.rcParams['figure.figsize'] = (20, 7)

## Configuration

In [ ]:
# Set your database path here
L3_DB_PATH = pathlib.Path('file_path')  # Modify this path
ORBIT_FILE = pathlib.Path('SWOT_science_orbit.nc')  # Orbit file to validate
ORBIT_FILE_ORIGINAL = pathlib.Path('~/workspace/OPEN_TOOLBOX/altimetry-search/altimetry/search/resources/SWOT_orbit.nc')
CYCLE = 7  # Cycle number

db = NetcdfFilesDatabaseSwotLRL3(str(L3_DB_PATH), enable_layouts=False)

## Helper Functions

In [ ]:
def _resample(arr, x_orig, x_new):
    """1-D linear interpolation."""
    return np.interp(x_new, x_orig, arr)

def read_real_pass(db, cycle_number, pass_number):
    """Query L3 database for one pass."""
    print(f'  Querying: cycle {cycle_number}, pass {pass_number}...')
    data = db.query(
        cycle_number=cycle_number,
        pass_number=pass_number,
        selected_variables=['time', 'longitude', 'latitude']
    )
    
    lon = data['longitude'].values
    lat = data['latitude'].values
    # L3 longitudes come in [0, 360]; convert to [-180, 180] so they
    # share the orbit.nc convention and overlay directly.
    lon = np.where(lon > 180, lon - 360, lon)
    
    return {
        'lon': lon,
        'lat': lat,
        'num_lines': lon.shape[0],
        'num_pixels': lon.shape[1],
    }

def read_orbit_pass(orbit_file, pass_idx):
    """Read one pass from orbit.nc."""
    print(f'  Reading: orbit.nc pass {pass_idx+1}...')
    ds = xr.open_dataset(orbit_file)
    
    lon_nadir = ds.lon_nadir.values[pass_idx, :]
    lat_nadir = ds.lat_nadir.values[pass_idx, :]
    left_lon = ds.left_polygon_lon.values[pass_idx, :]
    left_lat = ds.left_polygon_lat.values[pass_idx, :]
    right_lon = ds.right_polygon_lon.values[pass_idx, :]
    right_lat = ds.right_polygon_lat.values[pass_idx, :]
    
    ds.close()
    
    return {
        'lon_nadir': lon_nadir,
        'lat_nadir': lat_nadir,
        'left_lon': left_lon,
        'left_lat': left_lat,
        'right_lon': right_lon,
        'right_lat': right_lat,
    }

## Load Data

In [ ]:
PASS = 5 # Pass number (1-based)

In [ ]:
real_pass = read_real_pass(db, CYCLE, PASS)
orbit_pass = read_orbit_pass(ORBIT_FILE, PASS - 1)
orbit_pass_or = read_orbit_pass(ORBIT_FILE_ORIGINAL, PASS - 1)

## Full-Track Overlay

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

# Data
lon = real_pass['lon']
lat = real_pass['lat']

# Real L3 data (solid lines, high transparency)
ax.plot(lon[:, 0], lat[:, 0], 'b-', linewidth=3, label='Real L3 - Left', alpha=0.4)
ax.plot(lon[:, -1], lat[:, -1], 'r-', linewidth=3, label='Real L3 - Right', alpha=0.4)

# # Original Orbit NC data (dashed lines, high transparency)
ax.plot(orbit_pass_or['left_lon'], orbit_pass_or['left_lat'], 'y-', linewidth=3, label='OR Orbit NC - Left', alpha=0.4)
ax.plot(orbit_pass_or['right_lon'], orbit_pass_or['right_lat'], 'y-', linewidth=3, label='OR Orbit NC - Right', alpha=0.4)

# Orbit NC data (dashed lines, high transparency)
ax.plot(orbit_pass['left_lon'], orbit_pass['left_lat'], 'b--', linewidth=2, label='Orbit NC - Left', alpha=0.8)
ax.plot(orbit_pass['right_lon'], orbit_pass['right_lat'], 'r--', linewidth=2, label='Orbit NC - Right', alpha=0.8)

ax.set_xlabel('Longitude (°)', fontsize=12, fontweight='bold')
ax.set_ylabel('Latitude (°)', fontsize=12, fontweight='bold')
ax.set_title(f'Full Track Overlay - Cycle {CYCLE} Pass {PASS}\nReal L3 (solid, thin) vs Orbit NC (dashed, medium)', 
            fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=11, framealpha=0.95)

# ax.set_xlim([35, 40])    # Longitude
# ax.set_ylim([-50, -52.5])   # Latitude

# ax.set_xlim([120, 135])    # Longitude
# ax.set_ylim([-5, 5])   # Latitude
plt.tight_layout()
plt.show()

## Zooms (4 panels, including the equator)

Real L3 longitudes are converted to [-180, 180] in `read_real_pass`, so they overlay directly on the orbit.nc geometry.

Each panel is centred on the real nadir at a target latitude. The window is kept narrow, so any +-180 wrap stays outside the view.

In [ ]:
def plot_overlay(ax, real_pass, orbit_pass, orbit_pass_or):
    """Real L3 swath edges vs orbit.nc polygons on a single axis."""
    lon = real_pass['lon']
    lat = real_pass['lat']
    ax.plot(lon[:, 0],  lat[:, 0],  'b-',  linewidth=3, alpha=0.4, label='Real L3 - Left')
    ax.plot(lon[:, -1], lat[:, -1], 'r-',  linewidth=3, alpha=0.4, label='Real L3 - Right')
    ax.plot(orbit_pass_or['left_lon'],  orbit_pass_or['left_lat'],  'y-',  linewidth=3, alpha=0.4, label='OR Orbit NC - Left')
    ax.plot(orbit_pass_or['right_lon'], orbit_pass_or['right_lat'], 'y-',  linewidth=3, alpha=0.4, label='OR Orbit NC - Right')
    ax.plot(orbit_pass['left_lon'],  orbit_pass['left_lat'],  'b--', linewidth=2, alpha=0.8, label='Orbit NC - Left')
    ax.plot(orbit_pass['right_lon'], orbit_pass['right_lat'], 'r--', linewidth=2, alpha=0.8, label='Orbit NC - Right')
    ax.grid(True, alpha=0.3)

# Real nadir, used to locate the zoom centres
mid = real_pass['num_pixels'] // 2
lon_nadir = real_pass['lon'][:, mid]
lat_nadir = real_pass['lat'][:, mid]

# 4 zoom centres along the track (target latitudes), incl. the equator
HALF_DEG = 4.0
targets = [
    ('Equator',       0.0),
    ('South mid-lat', -40.0),
    ('North mid-lat',  40.0),
    ('High latitude',  float(np.nanmax(lat_nadir))),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
for ax, (name, target_lat) in zip(axes.ravel(), targets):
    idx  = int(np.nanargmin(np.abs(lat_nadir - target_lat)))
    clon = float(lon_nadir[idx])
    clat = float(lat_nadir[idx])
    plot_overlay(ax, real_pass, orbit_pass, orbit_pass_or)
    ax.set_xlim(clon - HALF_DEG, clon + HALF_DEG)
    ax.set_ylim(clat - HALF_DEG, clat + HALF_DEG)
    ax.set_title(f'{name}  (lat~{clat:.1f}deg, lon~{clon:.1f}deg)', fontweight='bold')
    ax.set_xlabel('Longitude (deg)')
    ax.set_ylabel('Latitude (deg)')

axes.ravel()[0].legend(loc='best', fontsize=9, framealpha=0.95)
fig.suptitle(f'Zooms - Cycle {CYCLE} Pass {PASS}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
